# Benchmarking AMAT NESTML models

Once we have completely validated the AMAT NESTML we can move onto benchmarking the model. Ensuring the NESTML ODE toolbox is processing this model is the most efficient way possible. Here we can build these large networks known as Brunel Balanced networks.  Brunel Balanced Network Model is a theoretical framework in computational neuroscience that describes how neural stability is maintained through the near-perfect cancellation of massive excitatory and inhibitory currents. This fluctuation-driven mechanism allows the model to replicate the asynchronous and irregular firing patterns observed in the cerebral cortex. By providing a unifying principle for E-I balance, the model helps explain network architecture, plasticity, and the emergence of coherent oscillations like gamma rhythms.

Source: https://www.bohrium.com/en/sciencepedia/feynman/keyword/brunel_balanced_network_model

NEST Tutorial: https://nest-simulator.readthedocs.io/en/v3.7/auto_examples/brunel_alpha_nest.html#id2


In [ ]:
import nest
import pynestml
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Mapping, Sequence
import numpy as np
import matplotlib.pyplot as plt
import importlib
from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils

SAVE_PATH = r"../juypter/report/spikes" 

# Build and validate the NESTML AMAT neuronal model 

In [ ]:
def build_nestml_model(
    source_path: Path,
    module_name: str,) -> tuple[str, str]:
    
    if not source_path.exists():
        raise FileNotFoundError(f"NESTML source not found: {source_path}")

    generated_module, generated_model = NESTCodeGeneratorUtils.generate_code_for( # nest call to compile nestml c++ script 
        str(source_path),
        module_name=module_name,
        logging_level="INFO",)

    try:
        nest.Install(generated_module)
        print(f"Successfully installed module: '{generated_module}'")
    except nest.kernel.NESTError:
        print(f"Module '{generated_module}' is already installed. Skipping installation step.")

    # Verify the MODEL name is loaded and recognized by the NEST engine
    if generated_model not in nest.Models():
        raise RuntimeError(
            f"Generation completed, but model '{generated_model}' is missing from nest.Models()."
        )
    
    # return module, name 
    return generated_module, generated_model

# defining global variables 
nest_model = "amat2_psc_exp"
nestml_source = Path("../neurons_nestml/amat_neuron.nestml")
nestml_module = "nestml_amat_module"
resolution_ms = 0.1

# calling build function
nestml_module, nestml_model = build_nestml_model(
    nestml_source,
    nestml_module,
)

print("Loaded module:", module_name)
print("Generated model:", nestml_model)
print("NESTML recordables:", nest.GetDefaults(nestml_model)["recordables"])

# Helper functions definitions 

In [ ]:
def identify_model_type(model_name: str, nestml_model_name: str) -> ModelType: 
    if model_name == nest_model:
        return "nest"   # classifying what model we are running for the specified sim
    if model_name == nestml_model_name:
        return "nestml"
    raise ValueError(f"Unsupported AMAT model: {model_name}")

def build_model_parameters(
    model_name: str,
    ModelType: ModelType,
    config: AMATConfig,
) -> dict[str, float]:
    """Translate shared conceptual parameters to each model's public API."""   # public API has a frozen set of params?
    defaults = nest.GetDefaults(model_name)
    tau_m = float(defaults["tau_m"])

    return {
        "tau_m": tau_m,
        "tau_v": config.gamma * tau_m, 
        "beta": config.B / tau_m,
        "alpha_1": config.alpha_1_mv,
        "alpha_2": config.alpha_2_mv,
        "omega": config.resting_threshold_mv,
    }

    

# NEST Parameter Config 

In [1]:
# restricts so only modeltype can be the following 
ModelType = Literal["nest", "nestml"]

@dataclass(frozen=True) # pass AMATConfig into both nestml and nest configs, ensures params are kept the same 
class AMATConfig:

    # input current config - do we need this? 
    pulse_amplitude_pa: float = 1000.0
    pulse_width_ms: float = 1.0

    # sim time 
    simulation_time_ms: float = 1000.0

    # hard code params instead of defaults 
    tau_m_ms: float = 10.0
    gamma: float = 1.5
    B: float = 4.0

    E_L_mv: float = -70.0
    resting_threshold_mv: float = -65.0

    exc_n: int = 750 # Number of excitatory neurons in brunel 
    inh_n: int = 250 # Number of inhibitory neurons in brunel

    # -- brunel kernel parameters --- 
    epsilon = float 0.1                       # connection probability
    CE = int(epsilon * NE)              # # excitatory inputs per neuron
    CI = int(epsilon * NI)              # # inhibitory inputs per neuron
    J_ex = float 0.15                         # excitatory synaptic weight, pA
    g = float 5.0                              # relative inhibitory strength (E/I balance)
    J_in = float(g * J_ex)                    # inhibitory strength * exc weight , how is this being processed in the nestml? 
    delay = float 1.5                         # synaptic delay, ms

    # adaptation strength parameters 
    alpha_1_mv: float = 5.0
    alpha_2_mv: float = 0.0
    @property # turns it into a read-only attribute 
    
    def tau_v_ms(self) -> float:
        """Voltage-dependent threshold timescale."""
        if self.gamma <= 0:
            raise ValueError("gamma must be greater than zero because tau_v = gamma * tau_m.")
        return self.gamma * self.tau_m_ms

    @property # Fixed: removed the extra leading space
    def beta_per_ms(self) -> float: # Fixed: removed the extra leading space
        """Convert dimensionless B to the model parameter beta."""
        return self.B / self.tau_m_ms

@dataclass # writes boilerplate code for the classes e.g., init, rep, eq
class SimulationResult:
    model_name: str
    model_type: ModelType
    events: Mapping
    spikes: Mapping
    parameters: dict[str, float]


# Build Brunel Balanced Network (NESTML) 

Currently this is built with 1000 neurons for init tests & ensure compute workflow works correctly before using more compute 

In [ ]:
nest.ResetKernel() # reset kernel, clean slate with any pre-existing param values etc. 
nest.SetKernelStatus({"resolution": resolution_ms})
nest.set_verbosity("M_ERROR")

config = AMATConfig() # establish amat config 
try:
    nest.Install(nestml_module)
except Exception as exc: # error handling incase nestml didn't run? 
    print(f"{exc}")

model_type = identify_model_type(model_name, nestml_model_name) # utilising helper functions defined in the cell above 
parameters = build_model_parameters(model_name, model_type, config)

neuron = nest.Create(model_name, params=parameters)
NE = config.exc_n
NI = config.inh_n
N_total = NE + NI

# create populations exc, inh
nodes_ex = nest.Create(nestml_model, NE, params=parameters)
nodes_in = nest.Create(nestml_model, NI, params=parameters)
nodes = nodes_ex + nodes_in

# whether we use the inh, exc spikes? or we can put a hyperpolarising / depolarising current... look at the literature 
receptor_types = nest.GetDefaults(nestml_model)["receptor_types"]
print("Available receptor types:", receptor_types)

RECEPTOR_EXC = receptor_types["EXC_SPIKES"]
RECEPTOR_INH = receptor_types["INH_SPIKES"]
RECEPTOR_STIM = receptor_types["I_STIM"]

# recurrent connections
nest.Connect(
    nodes_ex, nodes,
    conn_spec={"rule": "fixed_indegree", "indegree": CE},
    syn_spec={"weight": config.J_ex, "delay": config.delay, "receptor_type": RECEPTOR_EXC},
)
nest.Connect(
    nodes_in, nodes,
    conn_spec={"rule": "fixed_indegree", "indegree": CI},
    syn_spec={"weight": config.J_in, "delay": config.delay, "receptor_type": RECEPTOR_INH},
)

# stimulating noise 
rng = np.random.default_rng(seed=42)
stim_dt_ms = 1.0                                   # coarser than resolution_ms is fine
stim_times = np.arange(stim_dt_ms, config.simulation_time_ms, stim_dt_ms)
noise_mean_pa = 0.0
noise_std_pa = 250.0
stim_amplitudes = noise_mean_pa + noise_std_pa * rng.standard_normal(stim_times.size)

noise_stim = nest.Create("step_current_generator", params={
    "amplitude_times": stim_times,
    "amplitude_values": stim_amplitudes,
})
nest.Connect(noise_stim, nodes, syn_spec={"receptor_type": RECEPTOR_STIM}) # 

# spike recording of neurons 
spikes_ex = nest.Create("spike_recorder") # spike recording exc
spikes_in = nest.Create("spike_recorder") # spike recording inh
nest.Connect(nodes_ex, spikes_ex) # nodes, spikes conn
nest.Connect(nodes_in, spikes_in) #

# save spike times for later analysis after .
np.savez(SAVE_PATH, "spikes.npz",
         ex_times=ex_events["times"], ex_senders=ex_events["senders"],
         in_times=in_events["times"], in_senders=in_events["senders"])

print(f"Built Brunel network: NE={NE}, NI={NI}, CE={CE}, CI={CI}")


# Run simulation , produce a raster plot, input plot 

In [ ]:
# run the simulation
nest.Simulate(config.simulation_time_ms)

# gather spikes 
ex_events = spikes_ex.get("events")
in_events = spikes_in.get("events")

# --- plot: raster on top, noisy injected current on bottom ---
fig, (ax_raster, ax_input) = plt.subplots(
    2, 1, figsize=(10, 6), sharex=True,
    gridspec_kw={"height_ratios": [2.5, 1]},
)

ax_raster.plot(ex_events["times"], ex_events["senders"], ".", color="tab:blue", markersize=2, label="excitatory")
ax_raster.plot(in_events["times"], in_events["senders"], ".", color="tab:red", markersize=2, label="inhibitory")
ax_raster.set_ylabel("neuron id")
ax_raster.set_title("Brunel network activity (AMAT NESTML)")
ax_raster.legend(loc="upper right", markerscale=4)

ax_input.plot(stim_times, stim_amplitudes, color="black", linewidth=0.6)
ax_input.set_ylabel("I_stim (pA)")
ax_input.set_xlabel("time (ms)")
ax_input.set_xlim(0, config.simulation_time_ms)

plt.tight_layout()
plt.show()

n_spikes = len(ex_events["times"]) + len(in_events["times"])
rate_hz = 1000.0 * n_spikes / (N_total * config.simulation_time_ms)
print(f"Mean firing rate: {rate_hz:.2f} Hz")
